In [1]:
import matplotlib.pyplot as plt
import matplotlib
import csv
import numpy as np
import pandas as pd
import os
import re
import scipy.stats as ss
import rmgpy.constants
from rmgpy.kinetics.arrhenius import PDepArrhenius
from rmgpy.reaction import Reaction
from rmgpy.species import Species
from rmgpy.molecule import Molecule
from rmgpy.kinetics.arrhenius import Arrhenius
from rmgpy.data.rmg import RMGDatabase
from rmgpy import settings
from rmgpy.chemkin import load_chemkin_file
from rmgpy.rmg.pdep import PDepReaction
from arkane.input import load_input_file
from IPython.display import IFrame
import matplotlib.lines as mlines

%matplotlib inline

# Kinetics

In [2]:
#master list for kinetics 
master_dictionary_kinetics = {}

In [3]:
#families that were edited in this work 
families_modified = ['1+2_Cycloaddition', 
                     '1,2_Insertion_CO',
                     '1,3_Insertion_CO2',
                     '1,3_sigmatropic_rearrangement',
                     'F_Abstraction', 
                     'H_Abstraction', 
                     'R_Addition_COm',
                     'R_Addition_MultipleBond',
                     'XY_Addition_MultipleBond',
                     '1,2_Insertion_carbene',  # 51 found 
                     'CO_Disproportionation', 
                     'Disproportionation',
                     'Singlet_Carbene_Intra_Disproportionation', 
                     'halocarbene_recombination', 
                     'halocarbene_CO_dimerization']                   
                     

new_families = [                     #new families
                     'CO2_Elimination_From_Carboxylic_Acid',
                     'CO_CF_bond_dissociation', 
                     'Enol_Ether_Formation', 
                     'Lactone_Formation',
                     'PFAS_Hydrolysis',
                     'Perfluoroalkene_Formation'
]

In [4]:
thermolibs = [
    'NCSU_2025_revised',
    'C1_C2_Fluorine', #putting Siddha's as most trusted because Caroline used this thermo for calcs
    'PFCA_thermo',
    'Fluorine',
    'primaryThermoLibrary',
    'FFCM1(-)',
    'halogens',
    'CHOF_G4',
    'CHOCl_G4',
    'CHOBr_G4',
    'CHOFCl_G4',
    'CHOFBr_G4',
    'CHOFClBr_G4',
    'DFT_QCI_thermo',
    '2-BTP_G4',
    'thermo_DFT_CCSDTF12_BAC',
    'SulfurHaynes'
    ]

families = new_families+families_modified
database = RMGDatabase()
database.load(
    path = settings['database.directory'],
    thermo_libraries = thermolibs,  # Can add others if necessary
    kinetics_families = families,
    reaction_libraries = ['PFOA_unimolecular_decomp', "ANL_Brown_pdep", "ANL_Brown_pdep_2", 'Brown_difluoromethane_seed'],
    kinetics_depositories = ['training'],
)

In [5]:
#key phrases to look out for in the families 

F = 'Farina et al.'
PW = 'present work'
G = 'Guzman et al.'

key_phrases = {
    
    #1,2_Insertion_carbene
    'Training reaction from kinetics library: CF2BrCl': F, 
    'Calculated with Gaussian 16 using M062X with D3 dispersion and jun-cc-pVTZ basis set': F, 
    'Training reaction from kinetics library: HY_elim': F, 
    'Ea taken from Table': F, 
    'Could not locate TS with Gaussian 16 using M062X with D3 dispersion and jun-cc-pVTZ basis set': F, 


    #1,2_Insertion_CO
    #'Calculated with Gaussian 16 using M062X with D3 dispersion and jun-cc-pVTZ basis set': F, 
    'Training reaction from kinetics library: PFAS_HPL': PW, 

    #1,3_Insertion_CO2
    #'Training reaction from kinetics library: PFAS_HPL': PW, 

    #halocarbene_recombination
    #'Calculated with Gaussian 16 using M062X with D3 dispersion and jun-cc-pVTZ basis set': F, 
    # 'barrier = 71.944 kJ/mol': F, 
    # 'barrier = 12.781 kJ/mol': F, 
    # 'VRC-TST CASPT2 aug-cc-pVDZ': F, 

    #R_Addition_COm
    'Training reaction from kinetics library: PFAS_HPL_Test/R_Addition_COm/': PW, 
    'M062X/JUN-CC-PVTZ': F, 

    #R_Addition_MultipleBond
    # 'Training reaction from kinetics library: 2-BTP': F, 
    # 'Training reaction from kinetics library: YF': F, 
    # 'Training reaction from kinetics library: CH3Cl': F, 
    # 'Training reaction from kinetics library: CF2BrCl': F, 
    # 'Training reaction from kinetics library: autotst/R_Addition_MultipleBond': F, 
    # 'Training reaction from kinetics library: PFAS_HPL_Test/R_Addition_MultipleBond/': PW, 

    #Singlet_Carbene_Intra_Disproportionation
    'Training reaction from kinetics library: YF': F, 
    "A factor estimated": F, 

    #F_Abstraction 
    'Electronic structures done using ANL0 compound method': G, 

#     #XY_Addition_MultipleBond
#     'Ea from Table 3 in': F, 
#     'Training reaction from kinetics library: PFAS_HPL_Test/XY_Addition_MultipleBond/': PW, 
}

habstraction_key_phrases = {
    #H_Abstraction 
    'Training reaction from kinetics library: autotst/H_Abstraction': F, 
    'Calculated at m062x/cc-pVTZ level with AutoTST': F, 
    'Training reaction from kinetics library: CF2BrCl': F, 
    'Training reaction from kinetics library: CH3Cl': F, 
    'Electronic structures done using ANL0 compound method': G,
    'From NIST CH2F2 model': F, 
    'From NIST 2019 CH2F2 model': F, 
    'Training reaction from kinetics library: 2-BTP': F, 
    'Training reaction from kinetics library: YF': F, 
    'M. Lily et al. / Journal of Fluorine Chemistry 161 (2014) 51–59': F, 

}

halocarbene_recombination_key_phrases = {
    #halocarbene_recombination
    #'Calculated with Gaussian 16 using M062X with D3 dispersion and jun-cc-pVTZ basis set': F, 
    'barrier = 71.944 kJ/mol': F, 
    'barrier = 12.781 kJ/mol': F, 
    'VRC-TST CASPT2 aug-cc-pVDZ': F, 
}

ramb_key_phrases = {
    #R_Addition_MultipleBond
    'Training reaction from kinetics library: 2-BTP': F, 
    'Training reaction from kinetics library: YF': F, 
    'Training reaction from kinetics library: CH3Cl': F, 
    'Training reaction from kinetics library: CF2BrCl': F, 
    'Training reaction from kinetics library: autotst/R_Addition_MultipleBond': F, 
    'Training reaction from kinetics library: PFAS_HPL_Test/R_Addition_MultipleBond/': PW, 
}


xyamb_key_phrases = {
    #XY_Addition_MultipleBond
    'Ea from Table 3 in': F, 
    'Training reaction from kinetics library: PFAS_HPL_Test/XY_Addition_MultipleBond/': PW, 
    'Calculated with Gaussian 16 using M062X with D3 dispersion and jun-cc-pVTZ basis set': F, 
    
}

disproportionation_key_phrases = {
                #in short desc
                "From NIST 2019 CH2F2 model": F, 
                "From NIST CH2F2 model": F, 
                #in long desc
                "Training reaction from kinetics library: 2-BTP": F, 
                "Training reaction from kinetics library: CH3Cl": F, 
                "Training reaction from kinetics library: CF2BrCl": F, 
    
}

CO_disp_key_phrases = {
    'From NIST CH2F2 model': F, 
    'Training reaction from kinetics library: CF2BrCl': F, 
    'Training reaction from kinetics library: 2-BTP': F, 
    'Training reaction from kinetics library: CH3Cl': F, 
}    

expecting_dictionary = {
    '1+2_Cycloaddition': 30,
    '1,2_Insertion_CO': 26,
    '1,3_Insertion_CO2': 15,
    '1,3_sigmatropic_rearrangement': 8,
    'F_Abstraction': 43,
    'H_Abstraction': 687,
    'R_Addition_COm': 10,
    'R_Addition_MultipleBond': 58,
    'XY_Addition_MultipleBond': 33,
    '1,2_Insertion_carbene': 51,
    'CO_Disproportionation': 20,
    'Disproportionation': 124,
    'Singlet_Carbene_Intra_Disproportionation': 4,
    'halocarbene_recombination': 4,
    'halocarbene_CO_dimerization': 3,
    'CO2_Elimination_From_Carboxylic_Acid': 7,
    'CO_CF_bond_dissociation': 29,
    'Enol_Ether_Formation': 5,
    'Lactone_Formation': 17,
    'PFAS_Hydrolysis': 21,
    'Perfluoroalkene_Formation': 6,

}



In [6]:
family_of_interest = [
                      '1+2_Cycloaddition',            #30 found 
                     '1,2_Insertion_CO',             #26 found 
                     '1,3_Insertion_CO2',            #15 found 
                     '1,3_sigmatropic_rearrangement',  #8 found
                     'F_Abstraction',                  #43 found
                     'H_Abstraction',                  #687 found
                     'R_Addition_COm',                #10 found 
                     'R_Addition_MultipleBond',       #58 found 
                     'XY_Addition_MultipleBond',       #33 found
                     '1,2_Insertion_carbene',         # 51 found 
                     'CO_Disproportionation',           #20 found
                     'Disproportionation',              #124 found
                     'Singlet_Carbene_Intra_Disproportionation', #4 found 
                     'halocarbene_recombination',      #4 individually found 
                     'halocarbene_CO_dimerization',                   
                     


                     'CO2_Elimination_From_Carboxylic_Acid',
                     'CO_CF_bond_dissociation', 
                     'Enol_Ether_Formation', 
                     'Lactone_Formation',
                     'PFAS_Hydrolysis',
                     'Perfluoroalkene_Formation'

    
]

for fam in family_of_interest: 

    training_depo = database.kinetics.families[fam].get_training_depository()
    collected_reactions_for_this_family = []
    
    if fam not in new_families: 
        count = 0 
        if fam == 'halocarbene_recombination':
            kphrases = halocarbene_recombination_key_phrases
        elif fam == 'R_Addition_MultipleBond': 
            kphrases = ramb_key_phrases
        elif fam == 'XY_Addition_MultipleBond': 
            kphrases = xyamb_key_phrases
        else: 
            kphrases = key_phrases
            
    
        if fam == 'Disproportionation': 
            kphrases = disproportionation_key_phrases
            for tr in training_depo.entries.values():   
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.long_desc), None)
                if matched_phrase:
                    #print(tr.short_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1

                #test in short description
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.short_desc), None)
                if matched_phrase:
                    # print(tr.short_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1
    
        elif fam == 'H_Abstraction': 
            kphrases = habstraction_key_phrases 
            for tr in training_depo.entries.values(): 

                #test in long description
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.long_desc), None)
                if matched_phrase:
                    # print(tr.long_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1
                #test in short description
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.short_desc), None)
                if matched_phrase:
                    # print(tr.short_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1
        
                    
        elif fam == 'CO_Disproportionation': 
            kphrases = CO_disp_key_phrases
            for tr in training_depo.entries.values(): 
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.long_desc), None)
                if matched_phrase:
                    #print(tr.short_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1

                #test in short description
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.short_desc), None)
                if matched_phrase:
                    # print(tr.short_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1

        else: 
            for tr in training_depo.entries.values(): 
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.long_desc), None)
                if matched_phrase:
                    # print(tr.long_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1


                    
    if fam in new_families: 
        count = 0 
        for tr in training_depo.entries.values(): 
            collected_reactions_for_this_family.append([tr, PW])
            count+=1

    master_dictionary_kinetics[fam] = collected_reactions_for_this_family
    assert(len(collected_reactions_for_this_family)==count)
    print(f'{fam}: {count}')
    
assert expecting_dictionary=={k:len(v) for k,v in master_dictionary_kinetics.items()}

1+2_Cycloaddition: 30
1,2_Insertion_CO: 26
1,3_Insertion_CO2: 15
1,3_sigmatropic_rearrangement: 8
F_Abstraction: 43
H_Abstraction: 687
R_Addition_COm: 10
R_Addition_MultipleBond: 58
XY_Addition_MultipleBond: 33
1,2_Insertion_carbene: 51
CO_Disproportionation: 20
Disproportionation: 124
Singlet_Carbene_Intra_Disproportionation: 4
halocarbene_recombination: 4
halocarbene_CO_dimerization: 3
CO2_Elimination_From_Carboxylic_Acid: 7
CO_CF_bond_dissociation: 29
Enol_Ether_Formation: 5
Lactone_Formation: 17
PFAS_Hydrolysis: 21
Perfluoroalkene_Formation: 6


In [81]:
import yaml

output_data = {}

for fam, collected_rxn_data in master_dictionary_kinetics.items():
    output_data = []
    for training_reaction, source in collected_rxn_data:
        rxn_entry = {
            "equation": str(training_reaction),
            "rate-constant": {
            "A": str(training_reaction.data.A),
            "n": float(training_reaction.data.n.value),
            "Ea": str(training_reaction.data.Ea)}
        }
        output_data.append(rxn_entry)
    yaml_output = yaml.dump(output_data, sort_keys=False, default_flow_style=False)
    print(yaml_output)
    
    
    # Optionally save to file
    with open(f"./SI/yamls/{fam}_training_reactions.yaml", "w") as f:
        f.write(yaml_output)

- equation: CF2 + C2H2F2 <=> C3H2F4
  rate-constant:
    A: 10.1799 cm^3/(mol*s)
    n: 2.87159
    Ea: 40.5376 kJ/mol
- equation: CF2 + C2H4 <=> C3H4F2
  rate-constant:
    A: 20.7011 cm^3/(mol*s)
    n: 2.98446
    Ea: 43.7961 kJ/mol
- equation: CCl2 + C2H4 <=> C3H4Cl2
  rate-constant:
    A: 86.8219 cm^3/(mol*s)
    n: 2.97056
    Ea: 7.89502 kJ/mol
- equation: CF2 + C2F4 <=> C3F6
  rate-constant:
    A: 4.10298 cm^3/(mol*s)
    n: 3.07477
    Ea: 21.6954 kJ/mol
- equation: CBr2 + C2H4 <=> C3H4Br2
  rate-constant:
    A: 145.611 cm^3/(mol*s)
    n: 2.95653
    Ea: -0.108502 kJ/mol
- equation: CF2 + CH2O <=> C2H2F2O
  rate-constant:
    A: 48.1575 cm^3/(mol*s)
    n: 2.76922
    Ea: 37.8393 kJ/mol
- equation: C2F2O2 <=> CF2O + CO-2
  rate-constant:
    A: 2.58e+12 s^-1
    n: 0.3
    Ea: 9852.72 cal/mol
- equation: C2F2O2-2 <=> CO2 + CF2
  rate-constant:
    A: 1.17e+13 s^-1
    n: 0.33
    Ea: 19504.5 cal/mol
- equation: C3F4O2 <=> C2F4O + CO-2
  rate-constant:
    A: 3.29e+12 s^-1


In [76]:
float(training_reaction.data.n.value)

1.73

In [62]:
#make the output yaml file


for fam, collected_rxn_data in master_dictionary_kinetics.items():
    for [training_reaction, source] in collected_rxn_data:
        print(fam, training_reaction, training_reaction.data.A, training_reaction.data.n, training_reaction.data.Ea, source)
        

1+2_Cycloaddition CF2 + C2H2F2 <=> C3H2F4 10.1799 cm^3/(mol*s) 2.87159 40.5376 kJ/mol Farina et al.
1+2_Cycloaddition CF2 + C2H4 <=> C3H4F2 20.7011 cm^3/(mol*s) 2.98446 43.7961 kJ/mol Farina et al.
1+2_Cycloaddition CCl2 + C2H4 <=> C3H4Cl2 86.8219 cm^3/(mol*s) 2.97056 7.89502 kJ/mol Farina et al.
1+2_Cycloaddition CF2 + C2F4 <=> C3F6 4.10298 cm^3/(mol*s) 3.07477 21.6954 kJ/mol Farina et al.
1+2_Cycloaddition CBr2 + C2H4 <=> C3H4Br2 145.611 cm^3/(mol*s) 2.95653 -0.108502 kJ/mol Farina et al.
1+2_Cycloaddition CF2 + CH2O <=> C2H2F2O 48.1575 cm^3/(mol*s) 2.76922 37.8393 kJ/mol Farina et al.
1+2_Cycloaddition C2F2O2 <=> CF2O + CO-2 2.58e+12 s^-1 0.3 9852.72 cal/mol present work
1+2_Cycloaddition C2F2O2-2 <=> CO2 + CF2 1.17e+13 s^-1 0.33 19504.5 cal/mol present work
1+2_Cycloaddition C3F4O2 <=> C2F4O + CO-2 3.29e+12 s^-1 0.31 15935.4 cal/mol present work
1+2_Cycloaddition C3F4O2-2 <=> CO2 + C2F4-2 3.81e+12 s^-1 0.64 32162.9 cal/mol present work
1+2_Cycloaddition C4F6O2 <=> C3F6O + CO-2 4.74

In [211]:
#latex code for formatting rate data 
def format_A(value):
    # Use exponential notation for very large or very small numbers, 
    # otherwise use fixed 2 decimal places
    if value == 0:
        return "0.00"
    elif abs(value) >= 1e4 or abs(value) < 1e-2:
        return f"{value:.2e}"
    else:
        return f"{value:.2f}"


latex_rows = []
for fam, collected_rxn_data in master_dictionary_kinetics.items():
    for [training_reaction, source] in collected_rxn_data:
        rxn_str = str(training_reaction).replace('_', r'\_')
        fam_str = str(fam).replace('_', r'\_')
        source_str = str(source).replace('_', r'\_')

        # A = str(training_reaction.data.A).replace(' ', '\:').replace(f'{training_reaction.data.A.value}', f'{training_reaction.data.A.value:.2f}')
        A_value_str = format_A(training_reaction.data.A.value)
        A = f'{A_value_str} ${training_reaction.data.A.units}$'
        n = training_reaction.data.n.value_si
        Ea = training_reaction.data.Ea.value_si
        # rate_data = f"$A$ = ${A}$, $n$ = {n}, $E_a$ = {Ea:.3e}"

        latex_rows.append(f"{rxn_str} & {A} & {n:.2f} & {Ea:.3e} & {fam_str} & {source_str} \\\\")

latex_table = (
    "\\begin{footnotesize}\n"
    "\\begin{longtable}{p{5cm}p{2cm}p{1.3cm}p{2cm}p{2.5cm}p{2cm}}\n"
    "\\caption{Kinetic parameters for training reactions (SI units).} \\label{tab:training_reactions} \\\\\n"
    "\\hline\n"
    "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Family & Source \\\\\n"
    "\\hline\n"
    "\\endfirsthead\n"
    "\\multicolumn{6}{c}{\\tablename\\ \\thetable{} -- continued from previous page} \\\\\n"
    "\\hline\n"
    "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Family & Source \\\\\n"
    "\\hline\n"
    "\\endhead\n"
    "\\hline \\multicolumn{6}{r}{{Continued on next page}} \\\\\n"
    "\\endfoot\n"
    "\\hline\n"
    "\\endlastfoot\n"
    + "\n".join(latex_rows) +
    "\n\\end{longtable}\n"
    "\\end{footnotesize}"
)

print(latex_table)

\begin{footnotesize}
\begin{longtable}{p{5cm}p{2cm}p{1.3cm}p{2cm}p{2.5cm}p{2cm}}
\caption{Kinetic parameters for training reactions (SI units).} \label{tab:training_reactions} \\
\hline
Reaction & $A$ & $n$ & $E_a$ [J/mol] & Family & Source \\
\hline
\endfirsthead
\multicolumn{6}{c}{\tablename\ \thetable{} -- continued from previous page} \\
\hline
Reaction & $A$ & $n$ & $E_a$ [J/mol] & Family & Source \\
\hline
\endhead
\hline \multicolumn{6}{r}{{Continued on next page}} \\
\endfoot
\hline
\endlastfoot
CF2 + C2H2F2 <=> C3H2F4 & 10.18 $cm^3/(mol*s)$ & 2.87 & 4.054e+04 & 1+2\_Cycloaddition & Farina et al. \\
CF2 + C2H4 <=> C3H4F2 & 20.70 $cm^3/(mol*s)$ & 2.98 & 4.380e+04 & 1+2\_Cycloaddition & Farina et al. \\
CCl2 + C2H4 <=> C3H4Cl2 & 86.82 $cm^3/(mol*s)$ & 2.97 & 7.895e+03 & 1+2\_Cycloaddition & Farina et al. \\
CF2 + C2F4 <=> C3F6 & 4.10 $cm^3/(mol*s)$ & 3.07 & 2.170e+04 & 1+2\_Cycloaddition & Farina et al. \\
CBr2 + C2H4 <=> C3H4Br2 & 145.61 $cm^3/(mol*s)$ & 2.96 & -1.085e+02 & 1+2\

In [9]:
family_of_interest = [
                      # '1+2_Cycloaddition',            #30 found 
                     # '1,2_Insertion_CO',             #26 found 
                     # '1,3_Insertion_CO2',            #15 found 
                     # '1,3_sigmatropic_rearrangement',  #8 found
                     # 'F_Abstraction',                  #43 found
                     # 'H_Abstraction',                  #687 found
                     # 'R_Addition_COm',                #10 found 
                     # 'R_Addition_MultipleBond',       #58 found 
                     # 'XY_Addition_MultipleBond',       #33 found
                     # '1,2_Insertion_carbene',         # 51 found 
                     # 'CO_Disproportionation',           #20 found
                     # 'Disproportionation',              #124 found
                     # 'Singlet_Carbene_Intra_Disproportionation', #4 found 
                     # 'halocarbene_recombination',      #4 individually found 
                     # 'halocarbene_CO_dimerization',                   
                     


                     # 'CO2_Elimination_From_Carboxylic_Acid',
                     # 'CO_CF_bond_dissociation', 
                     # 'Enol_Ether_Formation', 
                     'Lactone_Formation',
                     # 'PFAS_Hydrolysis',
                     # 'Perfluoroalkene_Formation'

    
]

In [11]:
#latex code for formatting rate data 
def format_A(value):
    if value == 0:
        return "0.00"
    elif abs(value) >= 1e4 or abs(value) < 1e-2:
        return f"{value:.2e}"
    else:
        return f"{value:.2f}"

# Build a mapping of unique sources (excluding "present work") to reference numbers
source_to_number = {}
next_ref_number = 1
for fam, collected_rxn_data in master_dictionary_kinetics.items():
    for [training_reaction, source] in collected_rxn_data:
        source_str = str(source)
        if source_str != "present work" and source_str not in source_to_number:
            source_to_number[source_str] = next_ref_number
            next_ref_number += 1

latex_rows = []
for fam in family_of_interest:
    collected_rxn_data = master_dictionary_kinetics[fam]
    for [training_reaction, source] in collected_rxn_data:
        rxn_str = str(training_reaction).replace('_', r'\_')
        fam_str = str(fam).replace('_', r'\_')

        source_str = str(source)
        if source_str == "present work":
            source_display = "PW"
        else:
            source_display = f"[{source_to_number[source_str]}]"

        A_value_str = format_A(training_reaction.data.A.value)
        A = f'{A_value_str}'
        n = training_reaction.data.n.value_si
        Ea = training_reaction.data.Ea.value_si
        latex_rows.append(f"{rxn_str} & {A} & {n:.2f} & {Ea:.3e} & {source_display} \\\\")

footnote_text = (
    "The units of the $A$ factor depend on the order of the reaction: "
    "$s^{-1}$ for unimolecular (first-order) reactions, "
    "$cm^3/(mol \\cdot s)$ for bimolecular (second-order) reactions, "
    "and $cm^6/(mol^2 \\cdot s)$ for termolecular (third-order) reactions. "
    "PW stands for present work."
)

latex_table = (
    "\\begin{footnotesize}\n"
    "\\begin{longtable}{p{7cm}p{1.5cm}p{1.3cm}p{1.5cm}p{0.5cm}>{\centering\arraybackslash}p{2cm}}\n"
    "\caption{FAM Training reactions.}\\\n"
    "\\hline\n"
    "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Source \\\\\n"
    "\\hline\n"
    "\\endfirsthead\n"
    "\\multicolumn{6}{c}{\\tablename\\ \\thetable{} -- continued from previous page} \\\\\n"
    "\\hline\n"
    "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Source \\\\\n"
    "\hline\n"
    "\endhead\n"
    "\hline \multicolumn{6}{r}{{Continued on next page}} \\\n"
    "\endfoot\n"
    "\hline\n"
    "\multicolumn{6}{p{14cm}}{\small\textit{Note:} Units of $A$ depend on reaction order: $s^{-1}$ (unimolecular), $cm^3/(mol \cdot s)$ (bimolecular), $cm^6/(mol^2 \cdot s)$ (termolecular). PW = present work.} \\\n"
    "\endlastfoot\n"
    + "\n".join(latex_rows) +
    "\n\\end{longtable}\n"
    "\\end{footnotesize}"
)
print(latex_table)

# Print the reference key so you know which number maps to which source
print("\n% Reference key:")
for src, num in source_to_number.items():
    print(f"% [{num}] {src}")

\begin{footnotesize}
\begin{longtable}{p{7cm}p{1.5cm}p{1.3cm}p{1.5cm}p{0.5cm}>{\centeringrraybackslash}p{2cm}}
\caption{FAM Training reactions.}\
\hline
Reaction & $A$ & $n$ & $E_a$ [J/mol] & Source \\
\hline
\endfirsthead
\multicolumn{6}{c}{\tablename\ \thetable{} -- continued from previous page} \\
\hline
Reaction & $A$ & $n$ & $E_a$ [J/mol] & Source \\
\hline
\endhead
\hline \multicolumn{6}{r}{{Continued on next page}} \
\endfoot
\hline
\multicolumn{6}{p{14cm}}{\small	extit{Note:} Units of $A$ depend on reaction order: $s^{-1}$ (unimolecular), $cm^3/(mol \cdot s)$ (bimolecular), $cm^6/(mol^2 \cdot s)$ (termolecular). PW = present work.} \
\endlastfoot
C2HF3O2 <=> C2F2O2 + FH & 9.57e+11 & 0.34 & 2.220e+05 & PW \\
C3HF5O2 <=> C3F4O2 + FH & 8.93e+10 & 0.59 & 2.374e+05 & PW \\
C4HF7O2 <=> C4F6O2 + FH & 1.06e+10 & 0.85 & 2.332e+05 & PW \\
C5HF9O2 <=> C5F8O2 + FH & 7.32e+07 & 1.61 & 2.313e+05 & PW \\
C2H3FO2 <=> C2H2O2 + FH & 1.35e+11 & 0.49 & 2.108e+05 & PW \\
C3H5FO2 <=> C3H4O2 + FH & 

In [ ]:
#latex code for formatting rate data 
def format_A(value):
    if value == 0:
        return "0.00"
    elif abs(value) >= 1e4 or abs(value) < 1e-2:
        return f"{value:.2e}"
    else:
        return f"{value:.2f}"

# Build a mapping of unique sources (excluding "present work") to reference numbers
source_to_number = {}
next_ref_number = 1
for fam, collected_rxn_data in master_dictionary_kinetics.items():
    for [training_reaction, source] in collected_rxn_data:
        source_str = str(source)
        if source_str != "present work" and source_str not in source_to_number:
            source_to_number[source_str] = next_ref_number
            next_ref_number += 1


for fam, collected_rxn_data in master_dictionary_kinetics.items():

    latex_rows = []
    
    for [training_reaction, source] in collected_rxn_data:
        rxn_str = str(training_reaction).replace('_', r'\_')
        fam_str = str(fam).replace('_', r'\_')

        source_str = str(source)
        if source_str == "present work":
            source_display = "PW"
        else:
            source_display = f"[{source_to_number[source_str]}]"

        A_value_str = format_A(training_reaction.data.A.value)
        A = f'{A_value_str}'
        n = training_reaction.data.n.value_si
        Ea = training_reaction.data.Ea.value_si
        latex_rows.append(f"{rxn_str} & {A} & {n:.2f} & {Ea:.3e} & {fam_str} & {source_display} \\\\")

    footnote_text = (
        "The units of the $A$ factor depend on the order of the reaction: "
        "$s^{-1}$ for unimolecular (first-order) reactions, "
        "$cm^3/(mol \\cdot s)$ for bimolecular (second-order) reactions, "
        "and $cm^6/(mol^2 \\cdot s)$ for termolecular (third-order) reactions. "
        "PW stands for present work."
    )
    
    latex_table = (
        "\\begin{footnotesize}\n"
        "\\begin{longtable}{p{5cm}p{2cm}p{1.3cm}p{2cm}p{2.5cm}>{\\centering\\arraybackslash}p{2cm}}\n"
        f"\\caption{{Kinetic parameters for training reactions (SI units).\\footnote{{{footnote_text}}}}} \\label{{tab:training_reactions}} \\\\\n"
        "\\hline\n"
        "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Family & Source \\\\\n"
        "\\hline\n"
        "\\endfirsthead\n"
        "\\multicolumn{6}{c}{\\tablename\\ \\thetable{} -- continued from previous page} \\\\\n"
        "\\hline\n"
        "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Family & Source \\\\\n"
        "\\hline\n"
        "\\endhead\n"
        "\\hline \\multicolumn{6}{r}{{Continued on next page}} \\\\\n"
        "\\endfoot\n"
        "\\hline\n"
        "\\endlastfoot\n"
        + "\n".join(latex_rows) +
        "\n\\end{longtable}\n"
        "\\end{footnotesize}"
    )
    print(latex_table)

    # Print the reference key so you know which number maps to which source
    print("\n% Reference key:")
    for src, num in source_to_number.items():
        print(f"% [{num}] {src}")

# Libraries

In [5]:
database.kinetics.libraries.keys()

dict_keys(['PFOA_unimolecular_decomp', 'ANL_Brown_pdep', 'ANL_Brown_pdep_2', 'Brown_difluoromethane_seed'])

In [85]:
dir(entry.data)

['Pmax',
 'Pmin',
 'Tmax',
 'Tmin',
 '_Pmax',
 '_Pmin',
 '_Tmax',
 '_Tmin',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__ne__',
 '__new__',
 '__pyx_vtable__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '_pressures',
 'arrhenius',
 'change_rate',
 'comment',
 'discrepancy',
 'efficiencies',
 'fit_to_data',
 'get_cantera_efficiencies',
 'get_effective_collider_efficiencies',
 'get_effective_pressure',
 'get_rate_coefficient',
 'highPlimit',
 'is_identical_to',
 'is_pressure_dependent',
 'is_pressure_valid',
 'is_similar_to',
 'is_temperature_valid',
 'pressures',
 'set_cantera_kinetics',
 'solute',
 'to_html',
 'uncertainty']

In [84]:
for entry in database.kinetics.libraries['ANL_Brown_pdep'].entries.values(): 

    print(entry.data.)

AttributeError: 'rmgpy.kinetics.arrhenius.PDepArrhenius' object has no attribute 'to_cantera'

In [108]:
import yaml

output_data = {}

for lib_name, library in database.kinetics.libraries.items():
    output_data = []
    for entry in database.kinetics.libraries[lib_name].entries.values():


        if not isinstance(entry.data, PDepArrhenius): 
            rxn_entry = {
                "equation": str(entry),
                "rate-constant": {
                "A": str(entry.data.A),
                "n": float(entry.data.n.value),
                "Ea": str(entry.data.Ea)}
            }

        else: 
            rates_at_different_pressures = []
            pressure_units = entry.data.pressures.units
            for pressure, arrhenius in zip(entry.data.pressures.value, entry.data.arrhenius):
                
                rate = {"P": f"{pressure} {pressure_units}",
                    "A": str(arrhenius.A),
                    "n": float(arrhenius.n.value),
                    "Ea": str(arrhenius.Ea)}
                rates_at_different_pressures.append(rate)
        
                
            rxn_entry = {
                "equation": str(entry),
                "type": 'pressure-dependent-Arrhenius',
                "rate-constants": rates_at_different_pressures
            }            



        output_data.append(rxn_entry)
    yaml_output = yaml.dump(output_data, sort_keys=False, default_flow_style=False)
    print(yaml_output)
        
        
    # Optionally save to file
    with open(f"./SI/yamls/{lib_name}_library_reactions.yaml", "w") as f:
        f.write(yaml_output)

- equation: PF8acid <=> CF2s + PF7acid
  rate-constant:
    A: 1.16993e+13 s^-1
    n: 0.0
    Ea: 356505 J/mol
- equation: PF8acid <=> CF3r + rPF7acid
  rate-constant:
    A: 5.42053e+15 s^-1
    n: 0.0
    Ea: 363267 J/mol
- equation: PF8acid <=> PF2alkylr + rPF6acid
  rate-constant:
    A: 6.81764e+16 s^-1
    n: 0.0
    Ea: 345596 J/mol
- equation: PF8acid <=> PF3alkylr + rPF5acid
  rate-constant:
    A: 5.77293e+18 s^-1
    n: 0.0
    Ea: 345168 J/mol
- equation: PF8acid <=> PF4alkylr + rPF4acid
  rate-constant:
    A: 3.7357e+17 s^-1
    n: 0.0
    Ea: 335622 J/mol
- equation: PF8acid <=> PF5alkylr + rPF3acid
  rate-constant:
    A: 8.04493e+16 s^-1
    n: 0.0
    Ea: 349065 J/mol
- equation: PF8acid <=> PF6alkylr + rPF2acid
  rate-constant:
    A: 9.97238e+16 s^-1
    n: 0.0
    Ea: 338695 J/mol
- equation: PF8acid <=> CO2 + PF7-1-H
  rate-constant:
    A: 5.66565e+13 s^-1
    n: 0.0
    Ea: 301929 J/mol
- equation: PF8acid <=> CO2 + HF + PF7-1-ene
  rate-constant:
    A: 1.1132

In [105]:
entry.data.pressures.value[0]

0.0394

In [55]:
#latex code for formatting rate data 
def format_A(value):
    if value == 0:
        return "0.00"
    elif abs(value) >= 1e4 or abs(value) < 1e-2:
        return f"{value:.2e}"
    else:
        return f"{value:.2f}"


for lib_name, library in database.kinetics.libraries.items():

    #already done
    if lib_name in ['PFOA_unimolecular_decomp']: 
        continue
        
    print(f'%%%%%%%%%%%%%%%%%%%%%%%%%% {lib_name} %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%')
    latex_rows = []
    for entry in database.kinetics.libraries[lib_name].entries.values(): 

        rxn_str = str(entry).replace('_', r'\_')
        
        if not isinstance(entry.data, PDepArrhenius): 

            A_value_str = format_A(entry.data.A.value)
            n = entry.data.n.value_si
            Ea = entry.data.Ea.value_si
            latex_rows.append(f"{rxn_str} & {A_value_str} & {n:.2f} & {Ea:.3e} \\\\")

        else: 

            row = (
                f"\n{rxn_str}"
                r" & \makecell[l]{"
              
                
                )   
            
            for pressure, arrhenius in zip(entry.data.pressures.value, entry.data.arrhenius):

                A_value_str = format_A(arrhenius.A.value)
                n = arrhenius.n.value_si
                Ea = arrhenius.Ea.value_si

                
                pressure_rate_row = f"{pressure} atm: $A=${A_value_str}, $n=${n:.2f}, $E_a=${Ea:.3e}\\\\"
                row+=pressure_rate_row
                
            row+="} \\\\"
            latex_rows.append(row)
        
       

    footnote_text = (
        "The units of the $A$ factor depend on the order of the reaction: "
        "$s^{-1}$ for unimolecular (first-order) reactions, "
        "$cm^3/(mol \\cdot s)$ for bimolecular (second-order) reactions, "
        "and $cm^6/(mol^2 \\cdot s)$ for termolecular (third-order) reactions. "
        "PW stands for present work."
    )

    if not isinstance(entry.data, PDepArrhenius): 
        latex_table = (
            "\\begin{footnotesize}\n"
            "\\begin{longtable}{p{7cm}p{1.5cm}p{1.3cm}p{1.5cm}p{0.5cm}>{\centering\arraybackslash}p{2cm}}\n"
            "\caption{FAM Training reactions.}\\\n"
            "\\hline\n"
            "Reaction & $A$ & $n$ & $E_a$ [J/mol]\\\\\n"
            "\\hline\n"
            "\\endfirsthead\n"
            "\\multicolumn{6}{c}{\\tablename\\ \\thetable{} -- continued from previous page} \\\\\n"
            "\\hline\n"
            "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Source \\\\\n"
            "\hline\n"
            "\endhead\n"
            "\hline \multicolumn{6}{r}{{Continued on next page}} \\\n"
            "\endfoot\n"
            "\hline\n"
            "\multicolumn{6}{p{14cm}}{\small\textit{Note:} Units of $A$ depend on reaction order: $s^{-1}$ (unimolecular), $cm^3/(mol \cdot s)$ (bimolecular), $cm^6/(mol^2 \cdot s)$ (termolecular). PW = present work.} \\\n"
            "\endlastfoot\n"
            + "\n".join(latex_rows) +
            "\n\\end{longtable}\n"
            "\\end{footnotesize}"
            )
        print(latex_table)
    
    else: 
        latex_table = (
            "\\begin{footnotesize}\n"
            "\\begin{longtable}{p{8cm}p{5cm}}\n"
            "\caption{FAM Training reactions.}\\\n"
            "\\hline\n"
            "Reaction & Rate \\\\\n"
            "\\hline\n"
            "\\endfirsthead\n"
            "\\multicolumn{2}{c}{\\tablename\\ \\thetable{} -- continued from previous page} \\\\\n"
            "\\hline\n"
            "Reaction & Rate \\\\\n"
            "\hline\n"
            "\endhead\n"
            "\hline \multicolumn{2}{r}{{Continued on next page}} \\\n"
            "\endfoot\n"
            "\hline\n"
            "\multicolumn{2}{p{14cm}}{\small\textit{Note:} Units of $A$ depend on reaction order: $s^{-1}$ (unimolecular), $cm^3/(mol \cdot s)$ (bimolecular), $cm^6/(mol^2 \cdot s)$ (termolecular). PW = present work.} \\\n"
            "\endlastfoot\n"
            + "\n".join(latex_rows) +
            "\n\\end{longtable}\n"
            "\\end{footnotesize}"
            )
        print(latex_table)


%%%%%%%%%%%%%%%%%%%%%%%%%% ANL_Brown_pdep %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
\begin{footnotesize}
\begin{longtable}{p{8cm}p{5cm}}
\caption{FAM Training reactions.}\
\hline
Reaction & Rate \\
\hline
\endfirsthead
\multicolumn{2}{c}{\tablename\ \thetable{} -- continued from previous page} \\
\hline
Reaction & Rate \\
\hline
\endhead
\hline \multicolumn{2}{r}{{Continued on next page}} \
\endfoot
\hline
\multicolumn{2}{p{14cm}}{\small	extit{Note:} Units of $A$ depend on reaction order: $s^{-1}$ (unimolecular), $cm^3/(mol \cdot s)$ (bimolecular), $cm^6/(mol^2 \cdot s)$ (termolecular). PW = present work.} \
\endlastfoot

CHF + H <=> CH2F & \makecell[l]{0.1 atm: $A=$2.02e+14, $n=$-1.73, $E_a=$3.606e+04\\0.215 atm: $A=$7.29e+14, $n=$-1.78, $E_a=$2.696e+04\\0.464 atm: $A=$1.13e+16, $n=$-2.01, $E_a=$1.928e+04\\1.0 atm: $A=$6.83e+17, $n=$-2.39, $E_a=$1.384e+04\\2.15 atm: $A=$8.74e+19, $n=$-2.85, $E_a=$1.100e+04\\4.64 atm: $A=$8.45e+21, $n=$-3.27, $E_a=$1.023e+04\\10.0 atm: $A=$2.90e+23, $n=$-3.55, $E

In [27]:
print(database.kinetics.libraries['ANL_Brown_pdep'].entries[0])

CHF + H <=> CH2F


In [22]:
database.kinetics.libraries['ANL_Brown_pdep'].entries[0].data.arrhenius

[Arrhenius(A=(2.02e+14,'cm^3/mol/s'), n=-1.73, Ea=(36.06,'kJ/mol'), T0=(1,'K')),
 Arrhenius(A=(7.29e+14,'cm^3/mol/s'), n=-1.78, Ea=(26.96,'kJ/mol'), T0=(1,'K')),
 Arrhenius(A=(1.13e+16,'cm^3/mol/s'), n=-2.01, Ea=(19.28,'kJ/mol'), T0=(1,'K')),
 Arrhenius(A=(6.83e+17,'cm^3/mol/s'), n=-2.39, Ea=(13.84,'kJ/mol'), T0=(1,'K')),
 Arrhenius(A=(8.74e+19,'cm^3/mol/s'), n=-2.85, Ea=(11,'kJ/mol'), T0=(1,'K')),
 Arrhenius(A=(8.45e+21,'cm^3/mol/s'), n=-3.27, Ea=(10.23,'kJ/mol'), T0=(1,'K')),
 Arrhenius(A=(2.9e+23,'cm^3/mol/s'), n=-3.55, Ea=(10.69,'kJ/mol'), T0=(1,'K')),
 Arrhenius(A=(2.76e+24,'cm^3/mol/s'), n=-3.69, Ea=(11.67,'kJ/mol'), T0=(1,'K')),
 Arrhenius(A=(7.48e+24,'cm^3/mol/s'), n=-3.68, Ea=(12.74,'kJ/mol'), T0=(1,'K')),
 Arrhenius(A=(6.17e+24,'cm^3/mol/s'), n=-3.54, Ea=(13.51,'kJ/mol'), T0=(1,'K'))]

In [24]:
type(database.kinetics.libraries['ANL_Brown_pdep'].entries[0].data)

rmgpy.kinetics.arrhenius.PDepArrhenius

In [35]:
entry.data.pressures.value

array([  0.1  ,   0.215,   0.464,   1.   ,   2.15 ,   4.64 ,  10.   ,
        21.5  ,  46.4  , 100.   ])

In [37]:
type(entry.data.arrhenius)

list